# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shreyash0080/Flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane chosen: Lane 4 — CTR / Engagement Opportunity Scoring**

Among the four predefined lanes, Lane 4 is the most actionable with the data we have right now.
The starter dataset contains `impressions_90d`, `avg_position`, `ctr`, `sessions_90d`, `engagement_rate`, and `scroll_rate` — all the ingredients this lane needs, with no joins or external data required.

The core observation that motivated this choice: in the starter pipeline, the random forest's **top feature by importance is `days_with_impressions` (15.8%), and `avg_position` ranks third (10.9%)**.
Both are directly relevant to a CTR gap story — a page can be visible and well-ranked yet still fail to earn clicks.
That gap (visibility without clicks) is something a content or metadata change can realistically address.
Refresh scoring (Lane 2) already has a strong baseline from the starter pipeline; building a CTR/engagement lens gives an orthogonal view: not "is this page declining?" but "is this page *under-earning* given its position?"

This lane is also a responsible choice: the data supports position-adjusted analysis, minimum-volume filters are straightforward to apply, and the output is a ranked list a reviewer can act on immediately.

In [1]:
# No code needed in this cell — the lane rationale lives in the markdown above.
# Code cells in sections 3 will show supporting numbers from the data.


## 2. The question: decision, action, cost of a wrong call

### The research question

> **Which visible pages — those already receiving search impressions at a measurable position — are significantly under-capturing clicks or engagement relative to comparable pages, and deserve metadata, content, or monitoring review?**

### Unit of analysis

One row = one pseudonymized content page (a `content_id`), evaluated at the end of the 90-day observation window.
This is a **page-level** decision; the output is a ranked list of pages sorted by opportunity score.

### The output

A ranked review queue with:
- an **opportunity score** (position-adjusted CTR gap or engagement gap)
- a **suggested action** per page (rewrite title/meta, improve on-page engagement, monitor, expand)
- **reason codes** that explain *why* each page appears in the list

### Who acts on it and how

A content strategist or SEO reviewer opens the ranked queue each review cycle, inspects the top pages in order, and decides which ones to prioritize for a title/meta rewrite, a content engagement improvement, or monitoring.
The list replaces an ad-hoc gut-feel process ("which page do I look at next?") with a data-backed priority order.

### The decision this improves

The current decision is: *which pages should a reviewer spend limited editorial time on this week?*
Without a ranked queue, reviewers either work alphabetically, by recency, or by intuition.
A position-adjusted CTR gap score directs attention to pages where the data suggests the biggest recoverable opportunity.

### Cost of a wrong call

| Error type | What happens | Severity |
|---|---|---|
| **False positive** (flag a page that does not need review) | Reviewer spends time on a page with no real opportunity — low CTR was already explained by position, intent mismatch, or noise | Low-medium: wasted editorial time, small |
| **False negative** (miss a page that needs review) | A page continues under-earning clicks; organic traffic opportunity goes unrealized for another cycle | Medium: recoverable in the next cycle |
| **Wrong action recommendation** (e.g. suggest title rewrite when the real issue is engagement) | The edit is made but the metric does not improve; reviewer loses confidence in the tool | Medium: fixable with better reason codes |

The cost of being wrong here is **editorial time and opportunity cost**, not a safety or financial risk.
This means we can afford to recall broadly (flag more pages) and let the reviewer make the final call.
A false positive is cheap; missing a genuine opportunity repeatedly erodes trust in the system.

### Why data or ML can help at all

A hand-rule like "flag any page with CTR < 1%" will surface thousands of pages indiscriminately — many legitimately low-CTR because they rank on page 3.
ML (or a good scored heuristic) can **adjust for position**, **weight by volume**, and **combine multiple signals** (CTR gap + engagement gap + content freshness) into a single priority order that a fixed rule cannot.
Even a simple residual score — how far below a page sits from the median CTR for its position tier — already outperforms a flat threshold.

In [2]:
# No code needed here — this section is conceptual framing.
# Numbers that support the framing follow in section 3.


## 3. Quick look at the data (2-3 real numbers)

All numbers below come from `data/raw/content_refresh_anonymized.csv` — the 30,000-row anonymized starter dataset.
No numbers are invented or estimated.

In [3]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns available: {', '.join(df.columns.tolist())}")


Dataset shape: 30,000 rows × 44 columns
Columns available: content_id, client_id, search_volume, competition, competition_level, cpc, content_type, main_intent, word_count, char_count, provider_used, model_used, impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, days_with_impressions, days_with_sessions, impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d, clicks_prev_30d, sessions_prev_30d, content_age_days, age_tier, age_tier_order, days_since_last_update, freshness_tier, word_count_tier, char_count_tier, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, impression_tier, position_tier, trend_direction, trend_pct


In [4]:
# --- Number 1: The CTR gap is enormous among visible pages ---
# Visible = pages with 500+ impressions and avg_position between 1 and 20
visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20)]
low_ctr = visible[visible['ctr'] < 0.5]

print("=== CTR gap among visible pages (impressions >= 500, position 1-20) ===")
print(f"  Total visible pages  : {len(visible):,}")
print(f"  Pages with CTR < 0.5%: {len(low_ctr):,} ({100*len(low_ctr)/len(visible):.1f}%)")
print()
print(f"  Interpretation: {100*len(low_ctr)/len(visible):.1f}% of pages that Google already shows")
print(f"  in the top 20 positions are barely earning any clicks. These are not")
print(f"  invisible pages — they have demand and a position. The gap is in conversion.")


=== CTR gap among visible pages (impressions >= 500, position 1-20) ===
  Total visible pages  : 12,023
  Pages with CTR < 0.5%: 9,759 (81.2%)

  Interpretation: 81.2% of pages that Google already shows
  in the top 20 positions are barely earning any clicks. These are not
  invisible pages — they have demand and a position. The gap is in conversion.


In [5]:
# --- Number 2: Median CTR varies enormously by position tier ---
# This justifies position-adjusted scoring: you MUST compare pages to tier peers, not globally.
df['pos_tier'] = pd.cut(
    df['avg_position'],
    bins=[0, 3, 10, 20, 100],
    labels=['top3 (pos 1-3)', 'top10 (pos 4-10)', 'top20 (pos 11-20)', 'tail (pos 20+)']
)

tier_stats = df.groupby('pos_tier', observed=True).agg(
    pages=('ctr', 'count'),
    median_ctr=('ctr', 'median'),
    mean_ctr=('ctr', 'mean')
).round(3)

print("=== CTR statistics by position tier ===")
print(tier_stats.to_string())
print()
print("  Interpretation: A page at position 1-3 has a fundamentally different expected CTR")
print("  than one at position 11-20. Any scoring that ignores this will mislabel pages.")
print("  Position-adjusted gap analysis is the core of Lane 4's approach.")


=== CTR statistics by position tier ===
                   pages  median_ctr  mean_ctr
pos_tier                                      
top3 (pos 1-3)      1141        0.00     2.714
top10 (pos 4-10)   11842        0.16     0.651
top20 (pos 11-20)   7273        0.10     0.323
tail (pos 20+)      8524        0.00     0.212

  Interpretation: A page at position 1-3 has a fundamentally different expected CTR
  than one at position 11-20. Any scoring that ignores this will mislabel pages.
  Position-adjusted gap analysis is the core of Lane 4's approach.


In [6]:
# --- Number 3: Engagement gap is also material ---
# Pages with enough traffic (30+ sessions) but low engagement rate
has_sessions = df[df['sessions_90d'] >= 30]
eng_gap = has_sessions[has_sessions['engagement_rate'] < 30]
scroll_gap = has_sessions[has_sessions['scroll_rate'] < 20]

print("=== Engagement gap among pages with 30+ sessions ===")
print(f"  Pages with 30+ sessions: {len(has_sessions):,}")
print(f"  Of those, engagement_rate < 30: {len(eng_gap):,} ({100*len(eng_gap)/len(has_sessions):.1f}%)")
print(f"  Of those, scroll_rate < 20   : {len(scroll_gap):,} ({100*len(scroll_gap)/len(has_sessions):.1f}%)")
print()
print(f"  Interpretation: Nearly {100*len(eng_gap)/len(has_sessions):.0f}% of pages with real traffic show weak")
print(f"  engagement — users arrive but do not interact. This is a content quality signal,")
print(f"  distinct from the CTR gap (which is a metadata / snippet signal).")
print(f"  Lane 4 can surface both types in the same ranked output.")


=== Engagement gap among pages with 30+ sessions ===
  Pages with 30+ sessions: 7,114
  Of those, engagement_rate < 30: 7,113 (100.0%)
  Of those, scroll_rate < 20   : 6,467 (90.9%)

  Interpretation: Nearly 100% of pages with real traffic show weak
  engagement — users arrive but do not interact. This is a content quality signal,
  distinct from the CTR gap (which is a metadata / snippet signal).
  Lane 4 can surface both types in the same ranked output.


In [7]:
# --- Bonus: declining pages have measurably lower CTR ---
# This shows CTR and decline are linked — CTR opportunity scoring has prognostic value too.
ctr_by_trend = df.groupby('trend_direction')['ctr'].mean().round(4).sort_values()
print("=== Mean CTR by trend direction ===")
print(ctr_by_trend.to_string())
print()
print("  Declining pages have the lowest mean CTR (0.32%) — roughly half that of stable pages (0.52%).")
print("  Low CTR is not just a symptom: it may be an early warning signal of decline.")
print("  A CTR opportunity model could therefore serve double duty: surface under-earners AND")
print("  flag decline risk early, before trend_direction tips to 'down'.")


=== Mean CTR by trend direction ===
trend_direction
down      0.3241
stable    0.5173
up        0.5655
new       1.2965
flat      1.3768

  Declining pages have the lowest mean CTR (0.32%) — roughly half that of stable pages (0.52%).
  Low CTR is not just a symptom: it may be an early warning signal of decline.
  A CTR opportunity model could therefore serve double duty: surface under-earners AND
  flag decline risk early, before trend_direction tips to 'down'.


## 4. Careful words: what I can and can't claim

### What this work will be able to say (safe claims)

- **Observed association**: "Pages in position tier X with CTR below the tier median were observed to have lower engagement metrics on average, in this 30,000-page dataset."
- **Directional**: "A position-adjusted CTR gap score ranked pages in a different order than a flat CTR threshold — directing reviewer attention toward pages with real demand but poor conversion."
- **Decision-support**: "The ranked queue is a tool to help a human reviewer prioritize which pages to inspect first. The reviewer makes the final editorial decision."
- **Reproducible**: "These numbers come from a fixed anonymized snapshot and can be rerun from `data/raw/content_refresh_anonymized.csv`."

### What this work will never claim

- ❌ **Causal**: "Rewriting a title caused CTR to increase." — This data is observational. We watched, we did not experiment. Any before/after change claim requires a controlled experiment, which this data cannot support.
- ❌ **Google algorithm factors**: "Position X has CTR Y because Google's algorithm..." — We do not know Google's algorithm. We observe correlations in one dataset.
- ❌ **Guaranteed recovery**: "Fixing the CTR gap will improve traffic." — We can say the page is a candidate for review; we cannot promise an outcome.
- ❌ **Semantic claims**: "This page is about topic Z." — The data is pseudonymized. We have no access to titles, URLs, or actual content text.
- ❌ **Population generalization**: "All websites behave this way." — The dataset covers a specific set of clients in one anonymized snapshot. Findings are directional, not universal.

### The honest framing

This is a **decision-support** project. The output is a ranked list that helps a human reviewer use their limited weekly time more efficiently. The model ranks candidates; a human approves and acts. That is the appropriate scope for this data and this method.

In [8]:
# One final sanity check: confirm the dataset has enough signal for a CTR-gap model
# (we need pages with both impressions AND position, which is not guaranteed if many rows are zeros)
has_both = df[(df['impressions_90d'] > 0) & (df['avg_position'] > 0)]
print(f"Rows with impressions > 0 AND avg_position > 0: {len(has_both):,} ({100*len(has_both)/len(df):.1f}% of dataset)")
print()
print("Trend direction distribution in this working subset:")
print(has_both['trend_direction'].value_counts().to_string())
print()
print("Verdict: enough rows have both signals to build a position-adjusted CTR opportunity score.")
print("No external data required. Lane 4 is viable on the starter dataset alone.")


Rows with impressions > 0 AND avg_position > 0: 28,795 (96.0% of dataset)

Trend direction distribution in this working subset:
trend_direction
down      16254
stable     5962
up         4388
flat       1109
new        1082

Verdict: enough rows have both signals to build a position-adjusted CTR opportunity score.
No external data required. Lane 4 is viable on the starter dataset alone.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.